# networking

In [ ]:
///- --test static

open testing

In [ ]:
open rust.rust_operators

## rust

### reqwest_response

In [ ]:
nominal reqwest_response =
    `(
        rust_type
            `($'reqwest_Response')
            `($'reqwest_wasm::Response')
            "type reqwest_Response = class end"
    )

### reqwest_error

In [ ]:
nominal reqwest_error =
    `(
        rust_type
            `($'reqwest_Error')
            `($'reqwest_wasm::Error')
            "type reqwest_Error = class end"
    )

### request_builder

In [ ]:
type request_builder_switch =
    {
        Gleam : $'reqwest_RequestBuilder'
        Lua : $'reqwest_RequestBuilder'
        Fsharp : $'reqwest_RequestBuilder'
        Python : $'reqwest_RequestBuilder'
        TypeScript : $'reqwest_RequestBuilder'
        Cpp : $'reqwest_RequestBuilder'
        CppHost : $'reqwest_RequestBuilder'
        Delphi : $'reqwest_RequestBuilder'
        Cuda : $'reqwest_RequestBuilder'
        C : $'reqwest_RequestBuilder'
        Rust : $'reqwest_wasm::RequestBuilder'
    }
nominal request_builder =
    `(
        fsharp_global "type reqwest_RequestBuilder = class end"
        $'' : $'backend_switch `(request_builder_switch)'
    )

### request_type

In [ ]:
union request_type =
    | Get
    | Post

### request

In [ ]:
type request =
    {
        url : string
        request_type : request_type
        body : string
        json : bool
        auto_refresh : bool
    }

### new_request_get

In [ ]:
inl new_request_get (url : string) : request_builder =
    inl url = join url
    inl url = url |> sm'.to_std_string
    inl url = join url
    !\($'"reqwest_wasm::Client::builder().build().map_err(|err| err.to_string())?.get(!url)"')

### new_request_post

In [ ]:
inl new_request_post (url : string) : request_builder =
    inl url = join url
    inl url = url |> sm'.to_std_string
    inl url = join url
    !\($'"reqwest_wasm::Client::builder().build().map_err(|err| err.to_string())?.post(!url)"')

### request_send

In [ ]:
inl request_send (request : request_builder) : async.future_pin (resultm.result' reqwest_response reqwest_error) =
    inl request = join request
    !\($'"Box::pin(reqwest_wasm::RequestBuilder::send(!request))"')

### request_body

In [ ]:
inl request_body (body : string) (request : request_builder) : request_builder =
    inl body = body |> sm'.to_std_string
    !\\(body, $'"reqwest_wasm::RequestBuilder::body(!request, $0)"')

### request_header

In [ ]:
inl request_header (key : string) (value : string) (request : request_builder) : request_builder =
    inl request = join request
    inl key = key |> sm'.to_std_string
    inl value = value |> sm'.to_std_string
    !\\((key, value), $'"reqwest_wasm::RequestBuilder::header(!request, $0, $1)"')

### request_json

In [ ]:
inl request_json forall t. (obj : t) (request : request_builder) : request_builder =
    !\($'"reqwest_wasm::RequestBuilder::json(!request, &!obj)"')

### response_text

In [ ]:
inl response_text (response : reqwest_response) : async.future_pin (resultm.result' sm'.std_string reqwest_error) =
    !\($'"Box::pin(reqwest_wasm::Response::text(!response))"')

## fsharp

### tcp_client

In [ ]:
nominal tcp_client =
    `(
        backend_switch `(()) `({}) {
            Fsharp = (fun () =>
                global "type System_Net_Sockets_TcpClient = System.Net.Sockets.TcpClient\n"
                        ) : () -> ()
        }
        $'' : $'System_Net_Sockets_TcpClient'
    )

### new_tcp_client

In [ ]:
inl new_tcp_client () : tcp_client =
    run_target function
        | Fsharp => fun () => $'new `tcp_client ()'
        | _ => fun () => null ()

### ip_address

In [ ]:
nominal ip_address = $'System.Net.IPAddress'

### ip_address_parse

In [ ]:
inl ip_address_parse (s : string) : ip_address =
    s |> $'`ip_address.Parse'

### tcp_listener

In [ ]:
nominal tcp_listener = $'System.Net.Sockets.TcpListener'

### new_tcp_listener

In [ ]:
inl new_tcp_listener (ip_address : ip_address) (port : i32) : tcp_listener =
    $'new `tcp_listener (!ip_address, !port)'

### listener_start

In [ ]:
inl listener_start (listener : tcp_listener) : () =
    listener |> $'_.Start()'

### listener_stop

In [ ]:
inl listener_stop (listener : tcp_listener) : () =
    listener |> $'_.Stop()'

### client_connect_async

In [ ]:
inl client_connect_async
    (host : string)
    (port : i32)
    (ct : threading.cancellation_token)
    (client : tcp_client)
    : async.value_task
    =
    run_target function
        | Fsharp => fun () => $'!client.ConnectAsync (!host, !port, !ct)'
        | _ => fun () => null ()

## main

In [1]:
inl main () =
    init_trace_state None